# Agent 性能 PR 分析可视化

基于根目录 `full_analysis_distilled.csv`（1219 条 PR）展示：

- 变更规模 / 文件数 / 存活时间 / 评论数 与 **合并结果** 的关系
- 性能反模式、识别方式、优化层级、退化处置、修复引入新问题 等分布

> 运行前请在仓库根目录执行 `python3 generate_full_analysis.py` 以刷新 CSV（若 analysis JSON 有更新）。
> 图形默认保存至 `analysis_viz/figures/`。

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

ROOT = Path('..').resolve() if (Path('..') / 'full_analysis_distilled.csv').exists() else Path('.').resolve()
CSV_PATH = ROOT / 'full_analysis_distilled.csv'
FIG_DIR = Path('figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'font.family': 'DejaVu Sans',
})

df = pd.read_csv(CSV_PATH)
df['is_merged'] = (df['status'] == 'merged').astype(int)
df['outcome_label'] = df['status'].map({'merged': 'Merged', 'closed': 'Closed', 'open': 'Open'})

terminal = df[df['status'].isin(['merged', 'closed'])].copy()
print(f'Loaded {len(df)} PRs | terminal={len(terminal)} | merge rate={terminal.is_merged.mean():.1%}')
df.head(3)

In [ ]:
def savefig(name: str):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    print('saved', path)


def plot_merge_rate_by_bin(data, col, bins, labels, title, xlabel, fname, log_x=False):
    sub = data[[col, 'is_merged']].dropna().copy()
    sub['bin'] = pd.cut(sub[col], bins=bins, labels=labels, include_lowest=True)
    grp = sub.groupby('bin', observed=True).agg(n=('is_merged', 'size'), rate=('is_merged', 'mean'))

    fig, ax1 = plt.subplots(figsize=(9, 4.5))
    x = np.arange(len(grp))
    bars = ax1.bar(x, grp['rate'] * 100, color='#4C78A8', alpha=0.85, label='Merge rate (%)')
    ax1.set_ylim(0, 100)
    ax1.set_ylabel('Merge rate (%)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(grp.index.astype(str), rotation=0)
    ax1.set_xlabel(xlabel)
    ax1.set_title(title)
    ax1.axhline(terminal.is_merged.mean() * 100, color='#E45756', ls='--', lw=1.2, label='Overall terminal rate')

    ax2 = ax1.twinx()
    ax2.plot(x, grp['n'], color='#72B7B2', marker='o', lw=1.5, label='PR count')
    ax2.set_ylabel('PR count')

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)
    savefig(fname)
    plt.show()


def plot_box_by_outcome(data, col, title, xlabel, fname, log_y=False):
    sub = data[[col, 'outcome_label', 'status']].dropna()
    order = ['Merged', 'Closed', 'Open']
    groups = [sub.loc[sub.outcome_label == o, col].values for o in order if (sub.outcome_label == o).any()]
    labels = [o for o in order if (sub.outcome_label == o).any()]

    fig, ax = plt.subplots(figsize=(7, 4.5))
    bp = ax.boxplot(groups, tick_labels=labels, patch_artist=True, showfliers=False)
    colors = {'Merged': '#4C78A8', 'Closed': '#E45756', 'Open': '#F58518'}
    for patch, lab in zip(bp['boxes'], labels):
        patch.set_facecolor(colors.get(lab, '#ccc'))
        patch.set_alpha(0.7)
    if log_y:
        ax.set_yscale('log')
    ax.set_title(title)
    ax.set_xlabel('Outcome')
    ax.set_ylabel(xlabel)
    savefig(fname)
    plt.show()


def explode_pipe_col(data, col, exclude=None):
    exclude = set(exclude or [])
    counter = {}
    for val in data[col].fillna(''):
        for item in str(val).split('|'):
            item = item.strip()
            if not item or item in exclude:
                continue
            counter[item] = counter.get(item, 0) + 1
    return pd.Series(counter).sort_values(ascending=False)


def plot_bar_series(series, title, xlabel, fname, top_n=15, color='#4C78A8', horizontal=True):
    s = series.head(top_n).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(s))))
    if horizontal:
        ax.barh(s.index.astype(str), s.values, color=color, alpha=0.88)
        ax.set_xlabel(xlabel)
    else:
        ax.bar(s.index.astype(str), s.values, color=color, alpha=0.88)
        ax.set_ylabel(xlabel)
        plt.xticks(rotation=35, ha='right')
    ax.set_title(title)
    savefig(fname)
    plt.show()

## 1. 代码修改行数（changes）与合并结果

In [ ]:
plot_box_by_outcome(
    df, 'changes',
    'Code churn (changes) by outcome',
    'changes (additions + deletions)',
    '01_changes_boxplot.png',
    log_y=True,
)

plot_merge_rate_by_bin(
    terminal, 'changes',
    bins=[0, 100, 500, 2000, 10000, df['changes'].max() + 1],
    labels=['≤100', '101-500', '501-2k', '2k-10k', '>10k'],
    title='Merge rate by code-change size (terminal PRs)',
    xlabel='Changes bin',
    fname='02_changes_merge_rate.png',
)

## 2. 修改文件数与合并结果

In [ ]:
plot_box_by_outcome(
    df, 'file_count',
    'Modified file count by outcome',
    'file_count',
    '03_file_count_boxplot.png',
    log_y=True,
)

plot_merge_rate_by_bin(
    terminal, 'file_count',
    bins=[0, 1, 5, 20, 100, terminal['file_count'].max() + 1],
    labels=['1', '2-5', '6-20', '21-100', '>100'],
    title='Merge rate by modified file count (terminal PRs)',
    xlabel='Files bin',
    fname='04_file_count_merge_rate.png',
)

## 3. PR 存活时间与合并结果

In [ ]:
plot_box_by_outcome(
    df, 'lifespan_hours',
    'PR lifespan by outcome',
    'lifespan (hours)',
    '05_lifespan_boxplot.png',
    log_y=True,
)

plot_merge_rate_by_bin(
    terminal, 'lifespan_hours',
    bins=[0, 1, 24, 168, terminal['lifespan_hours'].max() + 1],
    labels=['<1h', '1-24h', '1-7d', '>7d'],
    title='Merge rate by PR lifespan (terminal PRs)',
    xlabel='Lifespan bin',
    fname='06_lifespan_merge_rate.png',
)

## 4. 评论规模与合并结果

In [ ]:
plot_box_by_outcome(
    df, 'comment_total',
    'Review + PR comment count by outcome',
    'comment_total',
    '07_comment_total_boxplot.png',
    log_y=False,
)

plot_merge_rate_by_bin(
    terminal, 'comment_total',
    bins=[-0.1, 0.5, 2.5, 9.5, terminal['comment_total'].max() + 1],
    labels=['0', '1-2', '3-9', '≥10'],
    title='Merge rate by comment volume (terminal PRs)',
    xlabel='Comments bin',
    fname='08_comment_merge_rate.png',
)

## 5. 标签分布：反模式 / 识别方式 / 优化层级 / 退化处置

In [ ]:
anti = explode_pipe_col(df, 'inefficiency_antipattern', exclude={'none', 'unknown'})
plot_bar_series(
    anti,
    'Inefficiency antipatterns (excluding none/unknown)',
    'PR count',
    '09_antipattern_bar.png',
    top_n=12,
    color='#E45756',
)

det = explode_pipe_col(df, 'detection_method', exclude={'unknown', ''})
plot_bar_series(
    det,
    'Maintainer detection methods (excluding unknown)',
    'PR count (multi-label)',
    '10_detection_method_bar.png',
    top_n=12,
    color='#54A24B',
)

opt = df['optimization_layer'].fillna('(missing)').value_counts()
plot_bar_series(
    opt,
    'Optimization layer distribution',
    'PR count',
    '11_optimization_layer_bar.png',
    top_n=15,
    color='#4C78A8',
)

reg = df['regression_handling'].fillna('(missing)').value_counts()
plot_bar_series(
    reg,
    'Regression / review-issue handling',
    'PR count',
    '12_regression_handling_bar.png',
    top_n=15,
    color='#B279A2',
)

## 6. 修复是否引入新问题（`antipattern_in_fix`）

In [ ]:
fix_col = df['antipattern_in_fix'].fillna('none').astype(str)
fix_col = fix_col.replace({'None': 'none', 'nan': 'none'})
fix_counts = fix_col.value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].pie(
    [fix_counts.get('none', 0), len(df) - fix_counts.get('none', 0)],
    labels=['none', 'non-none'],
    autopct='%1.1f%%',
    colors=['#D3D3D3', '#E45756'],
    startangle=90,
)
axes[0].set_title('Antipattern introduced in fix (binary)')

non_none = fix_counts.drop(labels=['none'], errors='ignore')
non_none = non_none[non_none.index != 'none']
if len(non_none):
    axes[1].barh(non_none.sort_values().index.astype(str), non_none.sort_values().values, color='#E45756', alpha=0.85)
    axes[1].set_title('Non-none antipattern_in_fix labels')
    axes[1].set_xlabel('PR count')
else:
    axes[1].text(0.5, 0.5, 'No non-none labels', ha='center', va='center')
    axes[1].axis('off')

savefig('13_antipattern_in_fix.png')
plt.show()

## 7. 补充：合并率、可复现性、Agent、边界类型

In [ ]:
# 7a Outcome pie (terminal)
fig, ax = plt.subplots(figsize=(5, 5))
vc = terminal['status'].value_counts()
ax.pie(vc.values, labels=[f"{k} ({v})" for k, v in vc.items()], autopct='%1.1f%%', startangle=90,
       colors=['#4C78A8', '#E45756'])
ax.set_title('Terminal outcomes (merged vs closed)')
savefig('14_outcome_pie.png')
plt.show()

# 7b Merge rate by agent (n>=30)
agent_stats = (
    df.groupby('agent')
    .agg(n=('pr_id', 'count'), merge_rate=('is_merged', 'mean'))
    .query('n >= 30')
    .sort_values('merge_rate', ascending=True)
)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(agent_stats.index, agent_stats['merge_rate'] * 100, color='#4C78A8', alpha=0.85)
ax.axvline(terminal.is_merged.mean() * 100, color='#E45756', ls='--', label='Terminal avg')
ax.set_xlabel('Merge rate (%)')
ax.set_title('Merge rate by agent (n≥30)')
ax.legend()
savefig('15_agent_merge_rate.png')
plt.show()

# 7c Reproducibility stacked with outcome
repro_cross = pd.crosstab(df['reproducibility'].fillna('unknown'), df['outcome_label'])
repro_cross = repro_cross.reindex(columns=[c for c in ['Merged', 'Closed', 'Open'] if c in repro_cross.columns])
fig, ax = plt.subplots(figsize=(8, 4.5))
repro_cross.plot(kind='bar', stacked=True, ax=ax, color=['#4C78A8', '#E45756', '#F58518'])
ax.set_title('Reproducibility rating vs outcome')
ax.set_xlabel('reproducibility')
ax.set_ylabel('PR count')
plt.xticks(rotation=0)
savefig('16_reproducibility_outcome.png')
plt.show()

# 7d Boundary tag
bnd = df['boundary_tag'].fillna('unknown').value_counts()
plot_bar_series(bnd, 'Capability boundary tag', 'PR count', '17_boundary_tag_bar.png', top_n=10, color='#72B7B2')

# 7e Review count vs merge rate
plot_merge_rate_by_bin(
    terminal, 'review_count',
    bins=[-0.1, 0.5, 1.5, 3.5, terminal['review_count'].max() + 1],
    labels=['0', '1', '2-3', '≥4'],
    title='Merge rate by formal review count',
    xlabel='Review count bin',
    fname='18_review_count_merge_rate.png',
)

print('All figures written to', FIG_DIR.resolve())